In [ ]:
REPO_URL = "https://github.com/YOUR_USER/Efficient-Models-course-ITMO-2026.git"
REPO_DIR = "/content/Efficient-Models-course-ITMO-2026"

import os, subprocess, sys

if not os.path.isdir(REPO_DIR):
    try:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
    except subprocess.CalledProcessError:
        os.makedirs(os.path.join(REPO_DIR, "src"), exist_ok=True)
        print("Clone не удался. Загрузите файлы из src/ в", os.path.join(REPO_DIR, "src"))
else:
    print("Репозиторий уже есть:", REPO_DIR)

os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
print("cwd:", os.getcwd())
print("src files:", sorted(os.listdir("src")) if os.path.isdir("src") else "НЕТ src/")

In [ ]:
import importlib.util, subprocess, sys

def ensure(pkg, import_name=None):
    name = import_name or pkg
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

ensure("triton")
ensure("transformers")
ensure("accelerate")
ensure("huggingface_hub")

import torch, triton
assert torch.cuda.is_available(), "Включите GPU: Runtime → Change runtime type → GPU"
major, minor = torch.cuda.get_device_capability()
print(f"torch {torch.__version__} | triton {triton.__version__} | CUDA {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)} (sm_{major}{minor})")
if major < 8:
    print("bf16 tensor cores нет — матмул пойдёт через fp32 (корректно, но медленнее).")

## Тесты ядер

In [ ]:
from tests import run_all_tests
run_all_tests()

## Бенчмарк ядер (общий)

In [ ]:
from benchmark import run_benchmarks
run_benchmarks()

## X16@W4ᵀ vs X16@W16ᵀ на размерах Llama-3.2-1B

M ∈ {128, 512, 2048}, формы весов как у `unsloth/Llama-3.2-1B-Instruct`.

In [ ]:
from bench_llama import run_bench
rows = run_bench()  # include_lm_head=True если нужен lm_head (долго)

## QuantizedLinear на Llama-3.2-1B-Instruct

Загрузка модели (~2.5 ГБ bf16) → замена `nn.Linear` на `QuantizedLinear` → dummy forward.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda"

tokenizer = AutoTokenizer.from_pretrained("unsloth/Llama-3.2-1B-Instruct")
model = AutoModelForCausalLM.from_pretrained(
    "unsloth/Llama-3.2-1B-Instruct",
    dtype=torch.bfloat16,
    device_map=device,
)
model.eval()

In [ ]:
import time

vocab = model.config.vocab_size

input_ids = torch.randint(0, min(vocab, 1000), (1, 16), device=device)
attention_mask = torch.ones_like(input_ids)

t0 = time.perf_counter()
out = model(input_ids=input_ids, attention_mask=attention_mask)
if torch.cuda.is_available():
    torch.cuda.synchronize()
dt = time.perf_counter() - t0
logits = out.logits

print(
    f"dummy forward: input_ids {tuple(input_ids.shape)} → logits {tuple(logits.shape)} "
    f"dtype={logits.dtype}  за {dt:.3f} с"
)

assert torch.isfinite(logits.float()).all(), "logits содержат NaN/Inf"

print("logits[0, -1, :5] =", logits[0, -1, :5].tolist())

In [ ]:
from quantize_llama import load_quantized_llama, run_dummy_forward
from linear import QuantizedLinear

model, tokenizer = load_quantized_llama(
    pack_bits=8,
    per_row=True,
    quantize_lm_head=True,
)

q_proj = model.model.layers[0].self_attn.q_proj
assert isinstance(q_proj, QuantizedLinear), type(q_proj)
print("q_proj[0]:", q_proj)

In [ ]:
vocab = model.config.vocab_size

input_ids = torch.randint(0, min(vocab, 1000), (1, 16), device=device)
attention_mask = torch.ones_like(input_ids)

t0 = time.perf_counter()
out = model(input_ids=input_ids, attention_mask=attention_mask)
if torch.cuda.is_available():
    torch.cuda.synchronize()
dt = time.perf_counter() - t0
logits = out.logits

print(
    f"dummy forward: input_ids {tuple(input_ids.shape)} → logits {tuple(logits.shape)} "
    f"dtype={logits.dtype}  за {dt:.3f} с"
)

assert torch.isfinite(logits.float()).all(), "logits содержат NaN/Inf"

print("logits[0, -1, :5] =", logits[0, -1, :5].tolist())